# Parte 1 – Optimización Numérica (Preguntas 3 y 4)

**Introducción a Redes Neuronales y Algoritmos Bioinspirados**  
Semestre 2026-01 · Universidad Nacional de Colombia

---

## Funciones de prueba elegidas

| Función | Dominio sugerido | Óptimo global |
|---------|-----------------|---------------|
| **Rosenbrock** | $[-5, 10]^d$ | $f(1,\ldots,1)=0$ |
| **Schwefel** | $[-500, 500]^d$ | $f(420.97,\ldots)\approx 0$ |

### Definiciones matemáticas

**Rosenbrock:**
$$f(\mathbf{x}) = \sum_{i=1}^{d-1}\left[100(x_{i+1}-x_i^2)^2 + (1-x_i)^2\right]$$

**Schwefel:**
$$f(\mathbf{x}) = 418.9829\,d - \sum_{i=1}^{d} x_i \sin\!\left(\sqrt{|x_i|}\right)$$

---

## Pregunta 3 – Métodos Heurísticos

Se optimizan ambas funciones en **2D** y **3D** usando:
1. Algoritmo Evolutivo (AE)
2. Optimización por Enjambre de Partículas (PSO)
3. Evolución Diferencial (DE)

## Pregunta 4 – Animaciones

Se generan GIFs que muestran el proceso de optimización (descenso por gradiente y método heurístico).


## 0. Instalación de dependencias

In [ ]:
# Instalar scipy si no está disponible (necesario para DE)
# !pip install scipy matplotlib numpy

## 1. Importaciones y funciones objetivo

In [ ]:
import numpy as np
import os
os.makedirs('../resultados', exist_ok=True)
import os
os.makedirs('../resultados', exist_ok=True)
import matplotlib
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.optimize import differential_evolution
import imageio
import os
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)  # reproducibilidad
print('Importaciones listas.')

In [ ]:
# ─────────────────────────────────────────────
# Funciones objetivo
# ─────────────────────────────────────────────

def rosenbrock(x):
    """Función de Rosenbrock. Óptimo global: f(1,...,1) = 0."""
    x = np.asarray(x, dtype=float)
    return np.sum(100.0 * (x[1:] - x[:-1]**2)**2 + (1 - x[:-1])**2)

def schwefel(x):
    """Función de Schwefel. Óptimo global: f(420.97,...) ≈ 0."""
    x = np.asarray(x, dtype=float)
    d = len(x)
    return 418.9829 * d - np.sum(x * np.sin(np.sqrt(np.abs(x))))

# Prueba rápida
print(f'Rosenbrock en (1,1)   = {rosenbrock([1.0, 1.0]):.6f}  (esperado: 0)')
print(f'Schwefel  en (420.97, 420.97) = {schwefel([420.9687, 420.9687]):.4f}  (esperado: ~0)')

## 2. Visualización de las superficies (2D)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Rosenbrock ──
x1 = np.linspace(-2, 2, 300)
x2 = np.linspace(-1, 3, 300)
X1, X2 = np.meshgrid(x1, x2)
Z_rosen = np.array([[rosenbrock([a, b]) for a, b in zip(row1, row2)]
                    for row1, row2 in zip(X1, X2)])
cp = axes[0].contourf(X1, X2, np.log1p(Z_rosen), levels=40, cmap='viridis')
plt.colorbar(cp, ax=axes[0], label='log(1+f)')
axes[0].set_title('Rosenbrock (2D) – escala log')
axes[0].set_xlabel('$x_1$'); axes[0].set_ylabel('$x_2$')
axes[0].plot(1, 1, 'r*', markersize=12, label='Óptimo (1,1)')
axes[0].legend()

# ── Schwefel ──
x = np.linspace(-500, 500, 300)
Xg, Yg = np.meshgrid(x, x)
Z_schw = np.array([[schwefel([a, b]) for a, b in zip(row1, row2)]
                   for row1, row2 in zip(Xg, Yg)])
cp2 = axes[1].contourf(Xg, Yg, Z_schw, levels=40, cmap='plasma')
plt.colorbar(cp2, ax=axes[1], label='f(x)')
axes[1].set_title('Schwefel (2D)')
axes[1].set_xlabel('$x_1$'); axes[1].set_ylabel('$x_2$')
axes[1].plot(420.9687, 420.9687, 'r*', markersize=12, label='Óptimo (~420.97)')
axes[1].legend()

plt.tight_layout()
plt.savefig('../resultados/superficies_2D.png', dpi=120)
plt.show()
print('Superficies guardadas.')

## 3. Algoritmo Evolutivo (AE)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Algoritmo Evolutivo – inspirado en el material de clase.
# Se guardan la población y el mejor valor por generación para las animaciones.
# ─────────────────────────────────────────────────────────────────────────────

def pob_inicial(N, d, LB, UB):
    return np.random.uniform(LB, UB, size=(N, d))

def evaluar_poblacion(Pob, f_obj):
    fitness = np.array([f_obj(ind) for ind in Pob])
    orden = np.argsort(fitness)
    return Pob[orden], fitness[orden]

def operacion_cruce(padre1, padre2):
    d = len(padre1)
    punto = np.random.randint(1, d)
    hijo1 = np.concatenate([padre1[:punto], padre2[punto:]])
    hijo2 = np.concatenate([padre2[:punto], padre1[punto:]])
    return hijo1, hijo2

def operacion_mutacion(individuo, LB, UB, prob_mut=0.3):
    nuevo = individuo.copy()
    for j in range(len(nuevo)):
        if np.random.rand() < prob_mut:
            nuevo[j] = np.random.uniform(LB[j], UB[j])
    return nuevo

def algoritmo_evolutivo(f_obj, d, LB, UB, N=50, max_gen=200,
                        fr_elite=0.2, fr_mut=0.15, semilla=None):
    """
    Algoritmo evolutivo con selección por ranking, cruce de un punto
    y mutación uniforme.

    Retorna:
        historia_pob  : lista de poblaciones por generación
        historia_best : array con el mejor valor por generación
        mejor_sol     : mejor solución encontrada
        n_eval        : número total de evaluaciones de la función objetivo
    """
    if semilla is not None:
        np.random.seed(semilla)

    LB, UB = np.array(LB), np.array(UB)
    n_elite = max(1, int(N * fr_elite))

    Pob = pob_inicial(N, d, LB, UB)
    historia_pob = []
    historia_best = []
    n_eval = 0

    for gen in range(max_gen):
        Pob, fitness = evaluar_poblacion(Pob, f_obj)
        n_eval += N
        historia_pob.append(Pob.copy())
        historia_best.append(fitness[0])

        # Élite
        nueva_pob = list(Pob[:n_elite])

        # Cruce para llenar el resto
        probs = np.arange(N, 0, -1, dtype=float)
        probs /= probs.sum()
        while len(nueva_pob) < N:
            idx1, idx2 = np.random.choice(N, 2, replace=False, p=probs)
            h1, h2 = operacion_cruce(Pob[idx1], Pob[idx2])
            nueva_pob.append(h1)
            if len(nueva_pob) < N:
                nueva_pob.append(h2)

        # Mutación
        nueva_pob = [operacion_mutacion(ind, LB, UB, fr_mut)
                     for ind in nueva_pob]
        # Preservar élite sin mutar
        nueva_pob[:n_elite] = list(Pob[:n_elite])

        Pob = np.array(nueva_pob)

    Pob, fitness = evaluar_poblacion(Pob, f_obj)
    n_eval += N
    historia_best.append(fitness[0])

    return historia_pob, np.array(historia_best), Pob[0], n_eval

print('Algoritmo Evolutivo definido.')

In [ ]:
# ── Correr AE en 2D y 3D para Rosenbrock y Schwefel ──

configs = [
    ('Rosenbrock', rosenbrock, 2, [-2, -1], [2, 3]),
    ('Rosenbrock', rosenbrock, 3, [-2, -1, -1], [2, 3, 3]),
    ('Schwefel',   schwefel,  2, [-500, -500], [500, 500]),
    ('Schwefel',   schwefel,  3, [-500, -500, -500], [500, 500, 500]),
]

resultados_ae = []
print(f"{'Función':12} {'Dim':4} {'Mejor valor':>14} {'Evaluaciones':>14}")
print('-' * 50)
for nombre, f, d, lb, ub in configs:
    hist_pob, hist_best, sol, n_eval = algoritmo_evolutivo(
        f, d, lb, ub, N=60, max_gen=300, semilla=42
    )
    resultados_ae.append((nombre, d, sol, hist_best[-1], n_eval, hist_pob, hist_best))
    print(f"{nombre:12} {d:4d} {hist_best[-1]:>14.4f} {n_eval:>14d}")

## 4. Optimización por Enjambre de Partículas (PSO)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# PSO – versión cognitiva + social clásica (Inertia Weight PSO)
# ─────────────────────────────────────────────────────────────────────────────

def pso(f_obj, d, LB, UB, N=40, max_iter=300,
        w=0.7, c1=1.5, c2=1.5, semilla=None):
    """
    Optimización por Enjambre de Partículas.

    Parámetros
    ----------
    w  : peso de inercia
    c1 : coeficiente cognitivo (atracción al mejor personal)
    c2 : coeficiente social (atracción al mejor global)

    Retorna historia de posiciones, mejor valor por iteración,
    mejor solución y número de evaluaciones.
    """
    if semilla is not None:
        np.random.seed(semilla)

    LB, UB = np.array(LB, dtype=float), np.array(UB, dtype=float)
    rango = UB - LB

    # Inicialización
    pos = np.random.uniform(LB, UB, (N, d))
    vel = np.random.uniform(-rango, rango, (N, d)) * 0.1
    pbest_pos = pos.copy()               # mejor posición personal
    pbest_val = np.array([f_obj(p) for p in pos])
    gbest_idx = np.argmin(pbest_val)
    gbest_pos = pbest_pos[gbest_idx].copy()
    gbest_val = pbest_val[gbest_idx]

    historia_pos  = [pos.copy()]
    historia_best = [gbest_val]
    n_eval = N

    for it in range(max_iter):
        r1 = np.random.rand(N, d)
        r2 = np.random.rand(N, d)
        vel = (w * vel
               + c1 * r1 * (pbest_pos - pos)
               + c2 * r2 * (gbest_pos - pos))
        pos = pos + vel
        # Recortar al dominio
        pos = np.clip(pos, LB, UB)

        vals = np.array([f_obj(p) for p in pos])
        n_eval += N

        mejoras = vals < pbest_val
        pbest_pos[mejoras] = pos[mejoras]
        pbest_val[mejoras] = vals[mejoras]

        g_idx = np.argmin(pbest_val)
        if pbest_val[g_idx] < gbest_val:
            gbest_val = pbest_val[g_idx]
            gbest_pos = pbest_pos[g_idx].copy()

        historia_pos.append(pos.copy())
        historia_best.append(gbest_val)

    return historia_pos, np.array(historia_best), gbest_pos, n_eval

print('PSO definido.')

In [ ]:
# ── Correr PSO en 2D y 3D ──

resultados_pso = []
print(f"{'Función':12} {'Dim':4} {'Mejor valor':>14} {'Evaluaciones':>14}")
print('-' * 50)
for nombre, f, d, lb, ub in configs:
    hist_pos, hist_best, sol, n_eval = pso(
        f, d, lb, ub, N=50, max_iter=300, semilla=42
    )
    resultados_pso.append((nombre, d, sol, hist_best[-1], n_eval, hist_pos, hist_best))
    print(f"{nombre:12} {d:4d} {hist_best[-1]:>14.4f} {n_eval:>14d}")

## 5. Evolución Diferencial (DE) – usando `scipy`

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Evolución Diferencial (DE) con scipy.optimize.differential_evolution
# Se rastrean las evaluaciones con un contador externo.
# ─────────────────────────────────────────────────────────────────────────────

def de_con_historia(f_obj, bounds, maxiter=300, popsize=15, seed=42):
    """
    Envuelve differential_evolution de scipy y registra el histórico
    del mejor valor por generación usando el callback.
    """
    historia_best = []
    contador = {'evals': 0}

    def f_wrapped(x):
        contador['evals'] += 1
        return f_obj(x)

    def callback(xk, convergence):
        historia_best.append(f_obj(xk))

    res = differential_evolution(
        f_wrapped, bounds,
        maxiter=maxiter, popsize=popsize,
        mutation=(0.5, 1.0), recombination=0.7,
        seed=seed, callback=callback, tol=1e-12
    )
    return np.array(historia_best), res.x, res.fun, contador['evals']

print('Evolución Diferencial definida.')

In [ ]:
# ── Correr DE en 2D y 3D ──

bounds_map = {
    ('Rosenbrock', 2): [(-2, 2), (-1, 3)],
    ('Rosenbrock', 3): [(-2, 2), (-1, 3), (-1, 3)],
    ('Schwefel',   2): [(-500, 500)] * 2,
    ('Schwefel',   3): [(-500, 500)] * 3,
}

resultados_de = []
print(f"{'Función':12} {'Dim':4} {'Mejor valor':>14} {'Evaluaciones':>14}")
print('-' * 50)
for nombre, f, d, lb, ub in configs:
    bounds = bounds_map[(nombre, d)]
    hist_best, sol, val, n_eval = de_con_historia(f, bounds)
    resultados_de.append((nombre, d, sol, val, n_eval, hist_best))
    print(f"{nombre:12} {d:4d} {val:>14.4f} {n_eval:>14d}")

## 6. Convergencia comparativa

In [ ]:
# ── Gráficas de convergencia por función y dimensión ──

funciones_dims = [('Rosenbrock', 2), ('Rosenbrock', 3),
                  ('Schwefel', 2), ('Schwefel', 3)]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, (fn, dim) in zip(axes.flat, funciones_dims):
    # AE
    r_ae = next(r for r in resultados_ae if r[0] == fn and r[1] == dim)
    r_pso = next(r for r in resultados_pso if r[0] == fn and r[1] == dim)
    r_de  = next(r for r in resultados_de  if r[0] == fn and r[1] == dim)

    ax.semilogy(r_ae[6],  label='AE',  color='steelblue')
    ax.semilogy(r_pso[6], label='PSO', color='darkorange')
    ax.semilogy(r_de[5],  label='DE',  color='green')
    ax.set_title(f'{fn} – {dim}D')
    ax.set_xlabel('Generación / Iteración')
    ax.set_ylabel('Mejor valor (escala log)')
    ax.legend()
    ax.grid(True, alpha=0.4)

plt.suptitle('Convergencia de métodos heurísticos', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../resultados/convergencia_heuristica.png', dpi=120)
plt.show()

## 7. Tabla resumen de resultados

In [ ]:
print('\n' + '='*70)
print(f"{'Método':6} | {'Función':10} | {'Dim':3} | {'Mejor valor':>14} | {'Evaluaciones':>12}")
print('='*70)

for r in resultados_ae:
    print(f"{'AE':6} | {r[0]:10} | {r[1]:3d} | {r[3]:>14.4f} | {r[4]:>12d}")
print('-'*70)
for r in resultados_pso:
    print(f"{'PSO':6} | {r[0]:10} | {r[1]:3d} | {r[3]:>14.4f} | {r[4]:>12d}")
print('-'*70)
for r in resultados_de:
    print(f"{'DE':6} | {r[0]:10} | {r[1]:3d} | {r[3]:>14.4f} | {r[4]:>12d}")
print('='*70)

## 8. Descenso por Gradiente (referencia para animación Q4)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Gradiente numérico y descenso por gradiente con condición inicial aleatoria
# (para contexto de la Pregunta 2, usado en la animación de Q4)
# ─────────────────────────────────────────────────────────────────────────────

def gradiente_numerico(f, x, eps=1e-5):
    """Gradiente numérico por diferencias finitas centradas."""
    x = np.array(x, dtype=float)
    grad = np.zeros_like(x)
    for i in range(len(x)):
        xp, xm = x.copy(), x.copy()
        xp[i] += eps; xm[i] -= eps
        grad[i] = (f(xp) - f(xm)) / (2 * eps)
    return grad

def descenso_gradiente(f, x0, lr=1e-4, max_iter=3000, tol=1e-8):
    """
    Descenso por gradiente con tasa de aprendizaje fija.
    Retorna historia de posiciones y valores.
    """
    x = np.array(x0, dtype=float)
    historia_x = [x.copy()]
    historia_f = [f(x)]
    n_eval = 1

    for _ in range(max_iter):
        g = gradiente_numerico(f, x)
        n_eval += 2 * len(x)
        x = x - lr * g
        val = f(x)
        n_eval += 1
        historia_x.append(x.copy())
        historia_f.append(val)
        if np.linalg.norm(g) < tol:
            break

    return np.array(historia_x), np.array(historia_f), n_eval

# ── Rosenbrock 2D ──
np.random.seed(7)
x0_ros = np.random.uniform([-2, -1], [2, 3])
hist_x_gd_ros, hist_f_gd_ros, ne_gd_ros = descenso_gradiente(
    rosenbrock, x0_ros, lr=2e-4, max_iter=5000
)
print(f'GD Rosenbrock 2D → f = {hist_f_gd_ros[-1]:.4f}, evals = {ne_gd_ros}')

# ── Schwefel 2D ──
x0_sch = np.random.uniform([-500, -500], [500, 500])
hist_x_gd_sch, hist_f_gd_sch, ne_gd_sch = descenso_gradiente(
    schwefel, x0_sch, lr=1.0, max_iter=3000
)
print(f'GD Schwefel  2D → f = {hist_f_gd_sch[-1]:.4f}, evals = {ne_gd_sch}')

## 9. Pregunta 4 – Animaciones GIF

Se generan cuatro GIFs:
1. Descenso por gradiente en Rosenbrock 2D
2. Descenso por gradiente en Schwefel 2D
3. PSO en Rosenbrock 2D
4. PSO en Schwefel 2D

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Generador de GIFs con imageio.v2 (estilo frame-a-frame)
# ─────────────────────────────────────────────────────────────────────────────

def _build_contour(f_obj, xlim, ylim, N_pts=200):
    """Pre-calcula la malla de contorno para reutilizarla en todos los frames."""
    x1v = np.linspace(*xlim, N_pts)
    x2v = np.linspace(*ylim, N_pts)
    X1g, X2g = np.meshgrid(x1v, x2v)
    Zg = np.array([[f_obj([a, b]) for a, b in zip(r1, r2)]
                   for r1, r2 in zip(X1g, X2g)])
    return X1g, X2g, Zg


def create_gif_gradiente(hist_x, hist_f, f_obj, xlim, ylim, titulo,
                         levels=50, cmap='viridis', filename='../resultados/gd_anim.gif',
                         n_frames=40, duration=120):
    """
    Genera un GIF del descenso por gradiente usando imageio.v2.
    Cada frame se renderiza como PNG temporal y luego se ensamblan.
    """
    print(f'Generando GIF: {filename}...')
    X1g, X2g, Zg = _build_contour(f_obj, xlim, ylim)

    total = len(hist_x)
    indices = np.unique(np.linspace(0, total - 1, n_frames, dtype=int))
    if indices[-1] != total - 1:
        indices = np.append(indices, total - 1)

    temp_files = []
    for k, i in enumerate(indices):
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
        fig.suptitle(titulo, fontsize=12, fontweight='bold')

        # ── Contorno + trayectoria ──
        ax1.contourf(X1g, X2g, np.log1p(Zg), levels=levels, cmap=cmap, alpha=0.85)
        ax1.set_xlim(xlim); ax1.set_ylim(ylim)
        ax1.set_xlabel('$x_1$'); ax1.set_ylabel('$x_2$')
        ax1.set_title(f'Iter {i}')
        if i > 0:
            path = hist_x[:i+1]
            ax1.plot(path[:, 0], path[:, 1], 'r--', alpha=0.5, linewidth=1.2)
        ax1.scatter(hist_x[i, 0], hist_x[i, 1], c='red', marker='o', s=60, zorder=5)

        # ── Curva de convergencia ──
        vals_hasta_i = hist_f[:i+1]
        ax2.semilogy(vals_hasta_i, 'b-', linewidth=1.8)
        ax2.set_xlim(0, total)
        ax2.set_ylim(max(hist_f[-1] * 0.1, 1e-10), hist_f[0] * 1.1 + 1)
        ax2.set_xlabel('Iteración'); ax2.set_ylabel('f(x) – escala log')
        ax2.set_title('Convergencia')
        ax2.grid(True, alpha=0.4)

        plt.tight_layout()
        tmp = f'../resultados/_tmp_gd_{k}.png'
        plt.savefig(tmp, dpi=100)
        plt.close(fig)
        temp_files.append(tmp)

    with imageio.v2.get_writer(filename, mode='I', duration=duration) as writer:
        for f in temp_files:
            writer.append_data(imageio.v2.imread(f))
    for f in temp_files:
        os.remove(f)
    print(f'GIF guardado: {filename}')


def create_gif_pso(historia_pos, hist_best, f_obj, xlim, ylim, titulo,
                   levels=50, cmap='plasma', filename='../resultados/pso_anim.gif',
                   n_frames=40, duration=150):
    """
    Genera un GIF del PSO (nube de partículas) usando imageio.v2.
    """
    print(f'Generando GIF: {filename}...')
    X1g, X2g, Zg = _build_contour(f_obj, xlim, ylim)

    total = len(historia_pos)
    indices = np.unique(np.linspace(0, total - 1, n_frames, dtype=int))
    if indices[-1] != total - 1:
        indices = np.append(indices, total - 1)

    temp_files = []
    best_vals = np.array(hist_best)
    for k, i in enumerate(indices):
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
        fig.suptitle(titulo, fontsize=12, fontweight='bold')

        # ── Contorno + partículas ──
        ax1.contourf(X1g, X2g, np.log1p(Zg + 1e-8), levels=levels, cmap=cmap, alpha=0.85)
        ax1.set_xlim(xlim); ax1.set_ylim(ylim)
        ax1.set_xlabel('$x_1$'); ax1.set_ylabel('$x_2$')
        ax1.set_title(f'Iter {i}  –  mejor f = {hist_best[i]:.4f}')
        pos2d = historia_pos[i][:, :2]
        ax1.scatter(pos2d[:, 0], pos2d[:, 1], c='white', marker='x', s=25, alpha=0.8)
        vals = np.array([f_obj(p) for p in historia_pos[i]])
        best_pos = historia_pos[i][np.argmin(vals)]
        ax1.scatter(best_pos[0], best_pos[1], c='red', marker='*', s=120, zorder=5)

        # ── Curva de convergencia ──
        ax2.semilogy(best_vals[:i+1], color='darkorange', linewidth=1.8)
        ax2.set_xlim(0, total)
        ax2.set_ylim(max(best_vals[-1] * 0.1, 1e-10), best_vals[0] * 1.1 + 1)
        ax2.set_xlabel('Iteración'); ax2.set_ylabel('Mejor f(x) – escala log')
        ax2.set_title('Convergencia PSO')
        ax2.grid(True, alpha=0.4)

        plt.tight_layout()
        tmp = f'../resultados/_tmp_pso_{k}.png'
        plt.savefig(tmp, dpi=100)
        plt.close(fig)
        temp_files.append(tmp)

    with imageio.v2.get_writer(filename, mode='I', duration=duration) as writer:
        for f in temp_files:
            writer.append_data(imageio.v2.imread(f))
    for f in temp_files:
        os.remove(f)
    print(f'GIF guardado: {filename}')

print('Funciones de animación (imageio) definidas.')

In [ ]:
# ── Generar GIFs de Descenso por Gradiente ──

create_gif_gradiente(
    hist_x_gd_ros, hist_f_gd_ros, rosenbrock,
    xlim=(-2.2, 2.2), ylim=(-1.2, 3.2),
    titulo='Descenso por Gradiente – Rosenbrock 2D',
    cmap='viridis', filename='../resultados/gd_rosenbrock_2D.gif'
)

create_gif_gradiente(
    hist_x_gd_sch, hist_f_gd_sch, schwefel,
    xlim=(-500, 500), ylim=(-500, 500),
    titulo='Descenso por Gradiente – Schwefel 2D',
    cmap='plasma', filename='../resultados/gd_schwefel_2D.gif'
)

In [ ]:
# ── Generar GIFs de PSO ──

r_pso_ros2 = next(r for r in resultados_pso if r[0]=='Rosenbrock' and r[1]==2)
r_pso_sch2 = next(r for r in resultados_pso if r[0]=='Schwefel'   and r[1]==2)

create_gif_pso(
    r_pso_ros2[5], r_pso_ros2[6], rosenbrock,
    xlim=(-2.2, 2.2), ylim=(-1.2, 3.2),
    titulo='PSO – Rosenbrock 2D',
    cmap='viridis', filename='../resultados/pso_rosenbrock_2D.gif'
)

create_gif_pso(
    r_pso_sch2[5], r_pso_sch2[6], schwefel,
    xlim=(-500, 500), ylim=(-500, 500),
    titulo='PSO – Schwefel 2D',
    cmap='plasma', filename='../resultados/pso_schwefel_2D.gif'
)

## 10. Discusión: Gradiente vs. Heurísticos

### ¿Qué aportó el descenso por gradiente?

- **Convergencia rápida** cuando el punto inicial está cerca del óptimo.
- **Eficiencia** en funciones unimodales: pocas evaluaciones para encontrar
  el mínimo local más cercano.
- **Limitaciones clave:**
  - Queda atrapado en **mínimos locales** (evidente en Schwefel y Rosenbrock con
    condición inicial desfavorable).
  - Necesita el gradiente (o su aproximación numérica), lo que encarece
    cada paso en alta dimensión.
  - Muy sensible a la **tasa de aprendizaje**: pequeña → lento; grande → diverge.

### ¿Qué aportaron los métodos heurísticos?

| Método | Fortaleza | Debilidad |
|--------|-----------|----------|
| **AE** | Explora con operadores de cruce y mutación, diversidad controlada por elitismo | Más evaluaciones que GD; sensible al tamaño de población |
| **PSO** | Memoria individual y social permite escapar de mínimos locales | Puede converger prematuro si `w` es muy pequeño |
| **DE** | Perturbaciones dirigidas por diferencias → muy competitivo en multimodales | Mayor número de parámetros a ajustar |

**Rosenbrock:** el gradiente llega al óptimo si parte cerca de él;
los heurísticos (especialmente DE) lo encuentran incluso desde lejos,
aunque con más evaluaciones.

**Schwefel:** la superficie tiene muchos mínimos locales y el gradiente
casi siempre queda atrapado. Los heurísticos —particularmente DE y PSO—
exploran más globalmente y obtienen soluciones considerablemente mejores
con un coste computacional razonable.

> **Conclusión:** ningún método domina en todos los casos.
> El gradiente es eficiente cuando la función es suave y unimodal;
> los heurísticos son más robustos ante multimodalidad pero requieren
> más evaluaciones. En la práctica se suelen combinar: un heurístico
> para encontrar una buena región y luego gradiente para refinar.

---
*Notebook generado como parte de la Tarea – Parte 1: Optimización Numérica.*  
*Código disponible en el repositorio Git del equipo.*